In [43]:
from matplotlib.gridspec import GridSpec
import networkx as nx
from itertools import product
from collections import defaultdict
import scipy as sp
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from scipy.stats import circmean
from scipy.stats import circstd
from pathlib import Path
import sys
import numpy as np
import glob
import os
from scipy.stats import iqr
from scipy.ndimage import gaussian_filter



%load_ext autoreload
%autoreload

def get_parent_dir():
    try:
        return Path(__file__).resolve().parent.parent
    except NameError:
        return Path.cwd().parent

parent_dir = str(get_parent_dir())
print("Parent directory:", parent_dir)

sys.path.append(parent_dir)
import models.load_state_patch
from stimulus_experiments.entrainment_experiments import *
from stimulus_experiments.entrainment_analysis import *
from stimulus_experiments.main_entrain import (
    parse_args,
    build_train_config,
    build_baseline_config,
    build_test_config,
)


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Parent directory: c:\Users\HP\ModellingProjects\Olivocerebellar-circuit


In [3]:


import brainpy as bp
import brainpy.helpers as helpers
import inspect

print("bp.load_state:", bp.load_state)
print("helpers.load_state:", helpers.load_state)

print("\nSource of bp.load_state:")
print(inspect.getsource(bp.load_state))

bp.load_state: <function load_state_fixed at 0x00000130EE3D7910>
helpers.load_state: <function load_state_fixed at 0x00000130EE3D7910>

Source of bp.load_state:
def load_state_fixed(target: DynamicalSystem, state_dict: Dict, **kwargs):
    """Copy parameters and buffers from :attr:`state_dict` into
    this module and its descendants.

    

    Args:
      target: DynamicalSystem. The dynamical system to load its states.
      state_dict: dict. A dict containing parameters and persistent buffers.

    Returns:
    -------
      ``NamedTuple``  with ``missing_keys`` and ``unexpected_keys`` fields:

      * **missing_keys** is a list of str containing the missing keys
      * **unexpected_keys** is a list of str containing the unexpected keys
    """
    # Map node names in state_dict to their base names
    state_by_base = {}
    for key in state_dict.keys():
        b = base_name(key)
        state_by_base.setdefault(b, []).append(key)


    nodes = target.nodes().subset(DynamicalSyst

- Breaking up runs into smaller runs with saving state in between
- Coupling saving and loading state effectively
- Test saving and loading state
- Write / change run_test


In [ ]:
# Test monitor preset setup

# Example: train run with specific options
argv = [
    "--seed", "0",
    "--run-type", "baseline",
    "--experiment", "nostim",
    "--simdur", "30_000",
    "--monitor-preset", "plasticity_min",
    "--parent-dir", "/home/izet/Olivocerebellar-circuit",
    "--tag", "monitor_test"
]

args = parse_args(argv)
config = build_baseline_config(args)

# Now call your simulation using the config
net, data = run_baseline(config)  # or simulate(config)

print(data.keys())

Baseline simulation time taken = 61.943278551101685 s
Saved baseline runner data to \home\izet\Olivocerebellar-circuit\results\stim_experiments_baseline_nostim_mon_plasticity_min__monitor_test\baseline_nostim_seed0_simdur30000.0.npz
Baseline saving time taken: 0.12005424499511719 s
dict_keys(['pfpc_weights', 'ts', 'PFPC_plasticity_on', 'OU_stim_pf_on', 'OU_stim_io_on', 'OU_stim_isi_mean', 'OU_stim_freq', 'OU_stim_start', 'OU_stim_amp_io_mean', 'OU_stim_amp_pf_mean', 'OU_stim_dur_io_mean', 'OU_stim_dur_pf_mean', 'monitor_preset', 'seed', 'dt', 'downsample', 'simdur', 'epoch_time', 'pf_pc_pre', 'pf_pc_post', 'io_pc_pre', 'io_pc_post'])


In [ ]:
# Test for saving and loading states
argv = [
    "--seed", "90",
    "--run-type", "train",
    "--experiment", "nostim",
    "--simdur", "35_000",
    "--monitor-preset", "plasticity_min",
    "--tag", "state_saving_test"
]

args = parse_args(argv)
config = build_train_config(args)


# Now call your simulation using the config
net, data, statepath = run_train(config)  

# Pick a key you care about, e.g. PurkinjeCell t_last_spike
vars_first_net = net.vars()
pc_key = [k for k in vars_first_net.keys()
          if "PurkinjeCell" in k and "V" in k][0]

print("Using key:", pc_key)

before_arr = vars_first_net[pc_key].value
print(f"First network {pc_key}, first 15:", before_arr[:15])


Converged at t=8000.0 ms (max Δw=0.00e+00)
Training simulation time taken = 12.097911596298218 s
Saving checkpoint into C:\Users\HP\ModellingProjects\Olivocerebellar-circuit\states\states_nostim__state_saving_test\nostim_isi120.0_isi_std0.0_seed90_simdur35000.0_state.bp
Saved training runner data to C:\Users\HP\ModellingProjects\Olivocerebellar-circuit\results\stim_experiments_train_nostim_mon_plasticity_min__state_saving_test\train_nostim_isi120.0_isi_std0.0seed90_simdur35000.0.npz
Training saving time taken: 0.020507097244262695 s
Using key: PurkinjeCell28.V
First network PurkinjeCell28.V, first 15: [-682.933     -57.986164  -61.442814  -61.583374  -64.7714    -52.716885
 -712.6992    -64.51682   -64.650536  -61.291553  -73.95031   -60.112087
  -69.57172   -63.201824  -62.069897]


In [33]:
argv = [
    "--seed", "66",
    "--run-type", "test",
    "--experiment", "nostim",
    "--simdur", "10_000",
    "--monitor-preset", "plasticity_min",
    "--parent-dir", "/home/izet/Olivocerebellar-circuit",
    "--pretraining-path", r"C:\Users\HP\ModellingProjects\Olivocerebellar-circuit\states\states_nostim__state_saving_test\nostim_isi120.0_isi_std0.0_seed0_simdur13000.0_state.bp",
    "--tag", "state_loading_test"
]

args = parse_args(argv)
config = build_test_config(args)



new_net, new_runner = init_net_and_runner(config['net_params'], seed= config['run_params']['seed'])
state = bc.load_pytree(config["pretraining_state_path"])


# 1. Inspect the top-level keys in the saved state
if isinstance(state, dict):
    print("Pretraining node names:", list(state.keys()))
else:
    # If it's some other mapping-like structure, inspect accordingly
    try:
        print("Dir of state:", dir(state))
    except Exception as e:
        print("Could not inspect state:", e)

# 2. Inspect names of nodes in the new network
nodes = new_net.nodes().subset(bp.DynamicalSystem)
print("New network node names:", list(nodes.keys()))

# Pick a key you care about, e.g. PurkinjeCell t_last_spike
vars_before = new_net.vars()
pc_key = [k for k in vars_before.keys()
          if "PurkinjeCell" in k and "V" in k][0]

print("Using key:", pc_key)

before_arr = vars_before[pc_key].value
print("Before load_state, first 10:", before_arr[:10])


# 3. Inspect the names of nodes after loading the state
result = bp.load_state(new_net, state)  # should print "[load_state_fixed] Called ..."
print("missing_keys:", result.missing_keys)
print("unexpected_keys:", result.unexpected_keys)
nodes_loaded_state = new_net.nodes().subset(bp.DynamicalSystem)
print("Network node names after reinstate:", list(nodes_loaded_state.keys()))


# Call your patched load_state
result = bp.load_state(new_net, state)
print("StateLoadResult:", result)
print("missing_keys:", result.missing_keys)
print("unexpected_keys:", result.unexpected_keys)

vars_after = new_net.vars()
after_arr = vars_after[pc_key].value
print("After load_state, first 10:", after_arr[:10])


Loading checkpoint from C:\Users\HP\ModellingProjects\Olivocerebellar-circuit\states\states_nostim__state_saving_test\nostim_isi120.0_isi_std0.0_seed0_simdur13000.0_state.bp
Pretraining node names: ['CerebellarNetwork2', 'PFBundles2', 'PurkinjeCell2', 'DeepCerebellarNuclei2', 'IONetwork2', 'PFtoPC_BCM2', 'PFtoPC2', 'PCToCN2', 'CNToIO2', 'IOToPC2', 'HalfWaveStimIOPF2', 'stimToPF2', 'stimToIO2', 'IONeuron2']
New network node names: ['CerebellarNetwork20', 'PFBundles20', 'PurkinjeCell20', 'DeepCerebellarNuclei20', 'IONetwork20', 'PFtoPC_BCM20', 'PFtoPC20', 'PCToCN20', 'CNToIO20', 'IOToPC20', 'HalfWaveStimIOPF20', 'stimToPF20', 'stimToIO20', 'IONeuron20']
Using key: PurkinjeCell20.V
Before load_state, first 10: [-69.66369  -71.09133  -70.03735  -69.73202  -70.81991  -71.08238
 -70.39149  -70.56257  -70.32716  -70.814316]
missing_keys: []
unexpected_keys: []
Network node names after reinstate: ['CerebellarNetwork20', 'PFBundles20', 'PurkinjeCell20', 'DeepCerebellarNuclei20', 'IONetwork20', 

In [28]:
# Test for running the test simulation with the loaded state

current_net_params = config['net_params']
run_params = config['run_params']
downsample = run_params['downsample']
duration = 1
net, runner, data = run_simulation(new_net, new_runner, duration, downsample)

vars_after = net.vars()
after_arr = vars_after[pc_key].value
print("After load_state, first 10:", after_arr[:10])

After load_state, first 10: [-690.1231    -58.87276   -62.120834  -61.68092   -62.66496   -53.226246
 -713.09906   -64.32118   -63.716446  -64.548065]


In [ ]:
print()

In [ ]:
# Plot weights over time
max_time = np.max(data['ts'])
print(max_time)
figure_2DEF(seed = run_params['seed'], mon= data,  save= True, save_dir = config['figures_dir'],   threshold = 2, xrange = [0.001, max_time-1])

In [5]:
# Test for restoring network

parent_dir = parent_dir
results_dir = os.path.join(
    parent_dir, "results", f"stim_experiments_test_for_test_{time.strftime('%m-%d_%H;%M;%S')}"
)
run_path = os.path.join(results_dir, f"test_test_runner.npz")

net_params = {"PFPC_plasticity_on": False, 
             "OU_stim_pf_on": True,
            "OU_stim_io_on": True,
}

run_params = { "seed": 88,
        "dt": 0.025,
        "simdur": 100_000,
        "downsample": 50, 
}

config = {
    'net_params': net_params,
    'run_params': run_params,
    'pretrain_snapshot_dir': snapshot_dir,
    'run_path':  run_path,

}

## Within run_test
new_net, new_runner = init_net_and_runner(net_params, seed= run_params['seed'])
print("Before - :", new_net.vars())

state, meta = load_snapshot(config['pretrain_snapshot_dir'])
bp.load_state(new_net, state)
print("After - :", new_net.vars())



print(state.items())


NameError: name 'snapshot_dir' is not defined